# STEP10. DMD annotation → 프레임 단위 눈 상태 GT

## 작업 목표

- DMD annotation을 프레임별 Open·Closed 정답으로 변환
- 영상과 annotation의 프레임 번호 확인
- 이진 분류에 사용할 라벨 기준 설정

## 입력·출력

| 구분 | 내용 |
|---|---|
| 입력 | DMD annotation JSON과 mosaic 영상 16쌍 |
| 출력 | 영상별 `frame_gt.csv`, `_summary.csv`, `_alignment.csv` |

## 순서

1. annotation 구조 확인
2. 영상과 annotation 프레임 정렬 확인
3. 16개 영상의 GT CSV 생성
4. 라벨 구성과 제외 비율 확인


## PART A — annotation 구조 확인

### 목적

- 라벨이 어떤 단위로 어떤 종류가 들어 있는지 실데이터로 확인한다.
- 이진 GT(Closed / Open)로 바꿀 때 무엇을 버릴지 정한다.

In [1]:
# [셀 1] 설정 · 경로 (import·경로·시드는 여기서 한 번만)

# --- 저장소 루트 부트스트랩 (모든 노트북 공통, 수정 금지) ---
import sys
from pathlib import Path

_anchors = []
if "__vsc_ipynb_file__" in globals():          # VS Code Notebook
    _anchors.append(Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
if globals().get("_dh"):                        # IPython 커널 시작 폴더
    _anchors.append(Path(globals()["_dh"][0]).resolve())
_anchors.append(Path.cwd().resolve())           # 최후 수단

# config.py 와 requirements.txt 를 '둘 다' 가진 폴더만 저장소 루트로 인정한다.
_root = next(
    (p for a in _anchors for p in [a, *a.parents]
     if (p / "config.py").is_file() and (p / "requirements.txt").is_file()),
    None,
)
if _root is not None:
    if str(_root) in sys.path:
        sys.path.remove(str(_root))
    sys.path.insert(0, str(_root))

import config

if _root is not None and Path(config.__file__).resolve().parent != _root:
    raise ImportError(f"의도하지 않은 config.py 가 import 되었습니다: {config.__file__}")
# --- 부트스트랩 끝 ---

# 프로젝트 모듈은 전부 src/ 에 평탄하게 둔다. 아직 패키지가 아니므로 sys.path 로 붙인다.
_src = str(config.PROJECT_ROOT / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

# config.setup_korean_font() 는 이 프로젝트 config.py 에 없다. 그림 라벨은 영문으로 쓴다.

import pandas as pd
import dmd_annotation as D
import build_dmd_eye_dataset as B

# 데이터셋 위치는 B.data_subdir 로 해석한다. data/raw/<name> 과 data/<name> 두
# 레이아웃을 모두 받아들여, 폴더를 옮겨도 노트북마다 경로를 고치지 않는다.
DMD_DIR = B.dmd_dir()
GT_DIR = config.OUTPUTS_DIR / "dmd_gt"
GT_DIR.mkdir(parents=True, exist_ok=True)

JSON_SUFFIX = "_rgb_ann_drowsiness.json"
AVI_SUFFIX = "_rgb_mosaic.avi"

jsons = sorted(DMD_DIR.glob(f"*{JSON_SUFFIX}"))
if not jsons:
    raise FileNotFoundError(f"annotation JSON 을 찾지 못했습니다: {config._rel(DMD_DIR)}")

print("annotation JSON :", len(jsons), "개")
print("출력 폴더       :", config._rel(GT_DIR))

annotation JSON : 16 개
출력 폴더       : outputs\dmd_gt


### 이진 라벨 기준

- `close` → 1, `open` → 0
- `opening`·`closing`·`undefined` → −1로 저장한 뒤 평가에서 제외
- 전이 상태는 Open 또는 Closed로 분명하게 나누기 어려워 제외했다.


In [2]:
# [셀 2] 한 영상 파싱 — 스키마와 라벨 종류 확인
import json
from collections import Counter

jp = str(jsons[0])
raw = json.load(open(jp, encoding="utf-8"))["openlabel"]

meta = D.video_meta(raw)
action_types = Counter(a["type"] for a in raw["actions"].values())
streams = {k: (v["stream_properties"].get("total_frames"),
               v["stream_properties"].get("sync", {}).get("frame_shift"))
           for k, v in raw["streams"].items()}

print("영상 :", Path(jp).name.replace(JSON_SUFFIX, ""))
print("스키마 :", raw["metadata"].get("schema_version"))
print("action 타입 :", dict(action_types))
print("stream (total_frames, frame_shift) :", streams)
print("메타 :", {k: meta[k] for k in
               ["ann_frames", "face_total_frames", "face_frame_shift",
                "gender", "age", "glasses", "setup", "weather"]})

df = pd.DataFrame(D.build_frame_table(raw))
print("\n프레임 표 shape :", df.shape)
print("eye_state 분포 :", df["eye_state"].value_counts().to_dict())
print("eye_gt_binary  :", df["eye_gt_binary"].value_counts().to_dict())
df.head(6)

영상 : gA_1_s5_2019-03-14T14;26;17+01;00
스키마 : 1.0.0
action 타입 : {'eyes_state/open': 1, 'eyes_state/close': 1, 'eyes_state/opening': 1, 'eyes_state/closing': 1, 'blinks/blinking': 1, 'yawning/Yawning with hand': 1, 'yawning/Yawning without hand': 1}
stream (total_frames, frame_shift) : {'face_camera': (5480, 0), 'body_camera': (5427, 54), 'hands_camera': (5407, 74)}
메타 : {'ann_frames': 5481, 'face_total_frames': 5480, 'face_frame_shift': 0, 'gender': 'Male', 'age': 47, 'glasses': True, 'setup': 'Car Stopped', 'weather': 'Rainy'}

프레임 표 shape : (5481, 7)
eye_state 분포 : {'open': 3596, 'opening': 778, 'close': 660, 'closing': 446, 'none': 1}
eye_gt_binary  : {0: 3596, -1: 1225, 1: 660}


,frame,eye_state,eye_closed,eye_gt_binary,is_blink,is_yawn,yawn_type
0,0,open,0,0,0,0,none
1,1,open,0,0,0,0,none
2,2,open,0,0,0,0,none
3,3,open,0,0,0,0,none
4,4,open,0,0,0,0,none
5,5,open,0,0,0,0,none


### 관찰 결과

- 라벨은 프레임이 아니라 **프레임 구간(action)** 단위이고, `eyes_state` / `blinks` / `yawning` 3계열이다.
- 카메라 스트림은 face / body / hands 3개이며 `frame_shift` 가 각각 0 / 54 / 74 다.
- `drowsy` · `microsleep` 라벨은 없다.
- 이 영상의 `eye_gt_binary` 는 open 3,596 / 제외 1,225 / close 660 이다.

### 목적

- annotation 프레임 번호를 mosaic 프레임 번호로 그대로 써도 되는지 확인한다.
- `face_camera.total_frames` 와 `frame_intervals` 가 1 어긋나는 경우가 있어 어느 쪽이 실제 프레임 수인지 확인한다.

In [3]:
# [셀 3] mosaic ↔ annotation 정렬 검증 (한 영상)
avi = Path(jp.replace(JSON_SUFFIX, AVI_SUFFIX))
mfc = D.mosaic_frame_count(str(avi)) if avi.exists() else None

print("annotation frames      :", meta["ann_frames"])          # frame_end + 1
print("face_camera total_frames:", meta["face_total_frames"])  # 스트림 메타데이터
print("mosaic frames (실측)    :", mfc)
print("정렬 :", mfc == meta["ann_frames"])

# face_total 이 1 작은 영상에서 마지막 프레임이 미라벨인지 확인한다.
# 미라벨이면 eye_gt_binary = -1 로 이미 제외되므로 정렬을 보정할 필요가 없다.
tail = df.tail(1)[["frame", "eye_state", "eye_gt_binary"]]
print("\n마지막 프레임 :", tail.to_dict("records"))

annotation frames      : 5481
face_camera total_frames: 5480
mosaic frames (실측)    : 5481
정렬 : True

마지막 프레임 : [{'frame': 5480, 'eye_state': 'none', 'eye_gt_binary': -1}]


## PART B — 16개 배치 변환

### 목적

- 16개 영상을 프레임 GT CSV 로 변환하고 영상·subject 메타와 정렬 결과를 표로 남긴다.
- 이 CSV 를 STEP11(눈 crop 생성)과 STEP13(프레임 단위 평가)의 정답 소스로 쓴다.

In [4]:
# [셀 4] 전체 배치 — GT CSV + 요약 + 정렬 리포트
def longest_run(flags) -> int:
    """연속 1의 최대 길이. microsleep 후보 구간 길이를 재기 위한 값."""
    best = cur = 0
    for v in flags:
        cur = cur + 1 if v else 0
        best = max(best, cur)
    return best


def count_events(flags) -> int:
    """0 -> 1 전환 횟수. 깜빡임을 프레임 수가 아니라 '횟수'로 세기 위함."""
    prev, n = 0, 0
    for v in flags:
        if v and not prev:
            n += 1
        prev = v
    return n


# 라벨 구성 집계도 이 루프에서 함께 모은다. 셀 5 가 JSON 16개를 다시 파싱하면
# 같은 계산을 두 번 하게 된다.
summary, align = [], []
comp, binc = Counter(), Counter()

for jp_i in map(str, jsons):
    base = Path(jp_i).name.replace(JSON_SUFFIX, "")
    ol = D.load_openlabel(jp_i)
    m = D.video_meta(ol)
    rows = D.build_frame_table(ol)
    pd.DataFrame(rows).to_csv(GT_DIR / f"{base}_frame_gt.csv", index=False)

    for r in rows:
        comp[r["eye_state"]] += 1
        binc[r["eye_gt_binary"]] += 1

    closed = [r["eye_closed"] for r in rows]
    blinks = [r["is_blink"] for r in rows]
    n = len(rows)

    avi_i = Path(jp_i.replace(JSON_SUFFIX, AVI_SUFFIX))
    mfc_i = D.mosaic_frame_count(str(avi_i)) if avi_i.exists() else None

    summary.append(dict(
        video=base, subject=base.split("_s5")[0], gender=m["gender"], age=m["age"],
        glasses=m["glasses"], setup=m["setup"], weather=m["weather"],
        ann_frames=m["ann_frames"], closed_frames=sum(closed),
        closed_pct=round(sum(closed) / n * 100, 1), max_closed_run=longest_run(closed),
        yawn_frames=sum(r["is_yawn"] for r in rows), blink_events=count_events(blinks)))
    align.append(dict(
        video=base, ann_frames=m["ann_frames"], face_total=m["face_total_frames"],
        face_frame_shift=m["face_frame_shift"], mosaic_frames=mfc_i,
        aligned=(mfc_i == m["ann_frames"]) if mfc_i else None))

sum_df = pd.DataFrame(summary)
al_df = pd.DataFrame(align)
sum_df.to_csv(GT_DIR / "_summary.csv", index=False)
al_df.to_csv(GT_DIR / "_alignment.csv", index=False)

print("저장 :", config._rel(GT_DIR))
print(f"정렬 성공 : {int(al_df['aligned'].sum())} / {len(al_df)}")
print(f"face_total 이 ann_frames 보다 1 작은 영상 : "
      f"{int((al_df.ann_frames - al_df.face_total == 1).sum())} 개")
sum_df[["video", "subject", "gender", "glasses", "setup", "ann_frames",
        "closed_frames", "closed_pct", "max_closed_run"]]

저장 : outputs\dmd_gt
정렬 성공 : 16 / 16
face_total 이 ann_frames 보다 1 작은 영상 : 4 개


,video,subject,gender,glasses,setup,ann_frames,closed_frames,closed_pct,max_closed_run
0,gA_1_s5_2019-03-14T14;26;17+01;00,gA_1,Male,True,Car Stopped,5481,660,12.0,60
1,gA_5_s5_2019-03-13T09;06;49+01;00,gA_5,Male,False,Car Stopped,5287,550,10.4,54
2,gB_10_s5_2019-03-12T10;35;20+01;00,gB_10,Male,False,Car Stopped,6055,1050,17.3,121
3,gB_10_s5_2019-03-13T14;17;28+01;00,gB_10,Male,False,Car Stopped,5417,614,11.3,76
4,gB_6_s5_2019-03-13T13;37;11+01;00,gB_6,Male,False,Car Stopped,5410,176,3.3,39
5,gB_7_s5_2019-03-13T13;55;52+01;00,gB_7,Male,False,Car Stopped,5405,616,11.4,98
6,gB_9_s5_2019-03-07T16;31;48+01;00,gB_9,Male,False,Car Stopped,5292,646,12.2,88
7,gC_13_s5_2019-03-12T10;03;00+01;00,gC_13,Female,False,Car Stopped,5380,165,3.1,42
8,gC_14_s5_2019-03-12T09;18;58+01;00,gC_14,Male,False,Car Stopped,5576,477,8.6,115
9,gE_29_s5_2019-03-15T13;51;09+01;00,gE_29,Female,True,Car Stopped,5363,953,17.8,81


> 영상별 subject 메타와 눈 감김 통계. `setup` 은 16개 모두 동일하고 `closed_pct` 는 3.1~17.8% 로 영상 간 차이가 크다. 두 값이 STEP11 의 split·표본 설계에 쓰인다.

In [5]:
# [셀 5] 프레임 라벨 구성과 평가 제외 비율 (셀 4 에서 누적한 집계를 그대로 쓴다)
N = sum(comp.values())
usable = binc[0] + binc[1]
print("16개 프레임 구성(%) :", {k: round(v / N * 100, 1) for k, v in comp.items()})
print(f"\n총 프레임        : {N:,}")
print(f"이진 GT 사용 가능 : {usable:,} ({usable/N*100:.1f}%)")
print(f"  close (=1)     : {binc[1]:,} ({binc[1]/N*100:.1f}%)")
print(f"  open  (=0)     : {binc[0]:,} ({binc[0]/N*100:.1f}%)")
print(f"제외 (=-1)       : {binc[-1]:,} ({binc[-1]/N*100:.1f}%)")
print(f"\n사용 가능 프레임 내 Closed 비율 : {binc[1]/usable*100:.1f}%"
      f"  (Closed:Open = 1 : {binc[0]/binc[1]:.1f})")
print("안경 착용 subject :",
      sorted(sum_df.loc[sum_df.glasses, "subject"].unique().tolist()))

16개 프레임 구성(%) : {'open': 64.0, 'closing': 11.4, 'opening': 14.3, 'close': 10.0, 'none': 0.0, 'undefined': 0.2}

총 프레임        : 88,226
이진 GT 사용 가능 : 65,302 (74.0%)
  close (=1)     : 8,827 (10.0%)
  open  (=0)     : 56,475 (64.0%)
제외 (=-1)       : 22,924 (26.0%)

사용 가능 프레임 내 Closed 비율 : 13.5%  (Closed:Open = 1 : 6.4)
안경 착용 subject : ['gA_1', 'gE_29', 'gZ_36']


### 결과

- 전체 88,226프레임 중 **65,302개(74.0%)**를 이진 평가에 사용했다.
- 사용 프레임 중 Closed는 **8,827개(13.5%)**였다.
- 영상 16개는 모두 `Car Stopped`, subject는 13명이다.
- 영상과 annotation의 프레임 정렬은 **16/16**에서 맞았다.


## 해석

- DMD annotation을 프레임별 Open·Closed GT로 변환했다. 별도 프레임 이동 보정은 필요하지 않았다.
- Closed 비율이 13.5%이므로 학습 표본을 만들 때 클래스 비율을 조정해야 한다.
- 이 GT는 눈 상태 기준이다. `drowsy`·`microsleep` 라벨이 없어 졸음 여부를 평가할 수는 없다.


## 한계

- 16개 영상이 모두 정차 상태에서 촬영됐다.
- 전이·미정 프레임 26.0%는 이진 평가에서 제외했다.
- subject 13명, 안경 착용자 3명으로 조건별 비교에는 표본이 적다.
- Open이 Closed보다 약 6.4배 많다.

## STEP10 요약

- DMD 16영상의 프레임별 눈 상태 GT를 만들었다.
- 사용 가능 프레임은 65,302개이며 Closed 비율은 13.5%다.
- 하품 라벨도 CSV에 남겼지만 이 노트북에서는 사용하지 않는다.
